# YOLO Iterative Training Pipeline - Metrics Analysis

This notebook visualizes training metrics over iterations to track model improvement.

In [ ]:
# Cell 1: Imports
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Cell 2: Load training history
history_path = Path("../logs/training_history.json")

if not history_path.exists():
    print(f"Error: {history_path} not found")
    print("Run at least one training iteration first.")
    df = pd.DataFrame()
else:
    with open(history_path) as f:
        history = json.load(f)
    
    # Convert to DataFrame
    df = pd.DataFrame(history)
    df['iteration'] = range(1, len(df) + 1)
    
    print(f"Loaded {len(df)} training iterations")
    print(f"\nLatest iteration:")
    print(df.iloc[-1])

In [ ]:
# Cell 3: Plot mAP50 progression (eval vs test)
if not df.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(df['iteration'], df['eval_map50'], marker='o', linewidth=2, 
            label='Eval mAP50', color='#2E86AB')
    ax.plot(df['iteration'], df['test_map50'], marker='s', linewidth=2,
            label='Test mAP50', color='#A23B72')
    
    ax.set_xlabel('Iteration', fontsize=12)
    ax.set_ylabel('mAP@0.5', fontsize=12)
    ax.set_title('Model Performance Over Iterations', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    # Add target line
    ax.axhline(y=0.85, color='green', linestyle='--', alpha=0.5, label='Target (0.85)')
    
    plt.tight_layout()
    plt.savefig('../logs/map50_progression.png', dpi=150)
    plt.show()
    
    # Summary stats
    print(f"\nEval mAP50: {df['eval_map50'].iloc[-1]:.3f} (target: 0.85)")
    print(f"Test mAP50: {df['test_map50'].iloc[-1]:.3f} (target: 0.82)")
    print(f"Improvement from iteration 1: {df['eval_map50'].iloc[-1] - df['eval_map50'].iloc[0]:.3f}")
else:
    print("No data to plot. Run training first.")

In [ ]:
# Cell 4: Plot F1 score progression
if not df.empty:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(df['iteration'], df['eval_f1'], marker='o', linewidth=2,
            label='Eval F1', color='#F18F01')
    ax.plot(df['iteration'], df['test_f1'], marker='s', linewidth=2,
            label='Test F1', color='#C73E1D')
    
    ax.set_xlabel('Iteration', fontsize=12)
    ax.set_ylabel('F1 Score', fontsize=12)
    ax.set_title('F1 Score Over Iterations', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    
    # Add target line
    ax.axhline(y=0.80, color='green', linestyle='--', alpha=0.5, label='Target (0.80)')
    
    plt.tight_layout()
    plt.savefig('../logs/f1_progression.png', dpi=150)
    plt.show()
    
    # Summary stats
    print(f"\nEval F1: {df['eval_f1'].iloc[-1]:.3f} (target: 0.80)")
    print(f"Test F1: {df['test_f1'].iloc[-1]:.3f} (target: 0.77)")
else:
    print("No data to plot. Run training first.")

In [ ]:
# Cell 5: Plot training data growth
if not df.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Image counts
    ax1.plot(df['iteration'], df['verified_count'], marker='o', linewidth=2,
             label='Verified', color='#06A77D')
    ax1.plot(df['iteration'], df['eval_count'], marker='s', linewidth=2,
             label='Eval', color='#D62246')
    ax1.set_xlabel('Iteration', fontsize=12)
    ax1.set_ylabel('Image Count', fontsize=12)
    ax1.set_title('Training Data Growth', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=11)
    ax1.grid(True, alpha=0.3)
    
    # Training time
    ax2.bar(df['iteration'], df['training_time_min'], color='#5E60CE', alpha=0.7)
    ax2.set_xlabel('Iteration', fontsize=12)
    ax2.set_ylabel('Training Time (minutes)', fontsize=12)
    ax2.set_title('Training Time Per Iteration', fontsize=14, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig('../logs/data_growth.png', dpi=150)
    plt.show()
    
    # Summary
    print(f"\nTotal verified images: {df['verified_count'].iloc[-1]}")
    print(f"Average training time: {df['training_time_min'].mean():.1f} minutes")
else:
    print("No data to plot. Run training first.")

In [ ]:
# Cell 6: Summary statistics
if not df.empty:
    print("=" * 60)
    print("TRAINING SUMMARY")
    print("=" * 60)
    
    print(f"\nTotal iterations: {len(df)}")
    print(f"Total verified images: {df['verified_count'].iloc[-1]}")
    print(f"Total training time: {df['training_time_min'].sum():.1f} minutes")
    
    print(f"\n--- Latest Model Performance ---")
    latest = df.iloc[-1]
    print(f"Eval  | mAP50: {latest['eval_map50']:.3f} | F1: {latest['eval_f1']:.3f} | "
          f"P: {latest['eval_precision']:.3f} | R: {latest['eval_recall']:.3f}")
    print(f"Test  | mAP50: {latest['test_map50']:.3f} | F1: {latest['test_f1']:.3f} | "
          f"P: {latest['test_precision']:.3f} | R: {latest['test_recall']:.3f}")
    
    print(f"\n--- Improvement from First Iteration ---")
    first = df.iloc[0]
    print(f"Eval mAP50: +{latest['eval_map50'] - first['eval_map50']:.3f}")
    print(f"Test mAP50: +{latest['test_map50'] - first['test_map50']:.3f}")
    print(f"Eval F1:    +{latest['eval_f1'] - first['eval_f1']:.3f}")
    print(f"Test F1:    +{latest['test_f1'] - first['test_f1']:.3f}")
    
    print(f"\n--- Success Criteria ---")
    eval_map_ok = "✓" if latest['eval_map50'] >= 0.85 else "✗"
    test_map_ok = "✓" if latest['test_map50'] >= 0.82 else "✗"
    eval_f1_ok = "✓" if latest['eval_f1'] >= 0.80 else "✗"
    test_f1_ok = "✓" if latest['test_f1'] >= 0.77 else "✗"
    
    print(f"{eval_map_ok} Eval mAP50 >= 0.85 (current: {latest['eval_map50']:.3f})")
    print(f"{test_map_ok} Test mAP50 >= 0.82 (current: {latest['test_map50']:.3f})")
    print(f"{eval_f1_ok} Eval F1 >= 0.80 (current: {latest['eval_f1']:.3f})")
    print(f"{test_f1_ok} Test F1 >= 0.77 (current: {latest['test_f1']:.3f})")
    
    # Production readiness
    verified = latest['verified_count']
    eval_map = latest['eval_map50']
    
    print(f"\n--- Production Readiness ---")
    if verified >= 700 and eval_map >= 0.88:
        print("Status: READY FOR AUTONOMOUS DEPLOYMENT")
        print("Recommendation: Export to production formats (ONNX, TensorRT)")
    elif verified >= 300 and eval_map >= 0.82:
        print("Status: READY FOR PRODUCTION WITH OVERSIGHT")
        print("Recommendation: Deploy with human verification")
    elif verified >= 100 and eval_map >= 0.75:
        print("Status: USEFUL FOR SCREENING")
        print("Recommendation: Continue annotating high-priority images")
    else:
        print("Status: EARLY STAGE")
        print("Recommendation: Clean more annotations (target: 100+ verified)")
    
    print("\n" + "=" * 60)
else:
    print("No training history found. Run at least one training iteration first.")
    print("\nTo start training:")
    print("  1. Clean annotations in X-AnyLabeling")
    print("  2. Save to data/working/")
    print("  3. File watcher auto-moves to data/verified/")
    print("  4. Training auto-triggers at 50 images (or run 'yolo-pipeline-train' manually)")